# Importing the Dataset without API

1. Import the needed libraries

We need to install gdown library to be able to list the files that are within the Google Drive folder
Gdown downloads a public file/folder from Google Drive. Gdown provides what curl/wget doesn't for Google Drive: Skip the security notice allowing you to download large files (curl/wget fails); Recursive download of files in a folder (maximum 50 files per folder).

In [ ]:
%pip install gdown

In [3]:
import gdown
import zipfile
import os
import shutil
import io


In [ ]:
# Step 1: Set the folder ID and construct the URL
folder_id = '1-9d5iNTJPv9VWla-j4PcCcza6igslBoj'  # Replace with folder ID
url = f'https://drive.google.com/drive/folders/{folder_id}'

# Step 2: Define the download and extraction folders
download_folder = 'downloaded_files'  # Folder to store downloaded .mglyph files
extracted_folder = 'extracted_files'  # Folder to store extracted files

# Create the folders if they don't exist
os.makedirs(download_folder, exist_ok=True)
os.makedirs(extracted_folder, exist_ok=True)

# Step 3: Download the entire folder using gdown
print("Downloading folder...")
gdown.download_folder(url, output=download_folder, quiet=True, use_cookies=False)
print("Download complete.")

# Step 4: Iterate through the downloaded .mglyph files and extract them
for root, dirs, files in os.walk(download_folder):
    for filename in files:
        if filename.endswith('.mglyph'):
            file_path = os.path.join(root, filename)
            extracted_path = os.path.join(extracted_folder, filename.replace('.mglyph', ''))  
            
            if not os.path.exists(extracted_path):  
                print(f"Extracting: {filename}")
                
                # Extract the contents
                try:
                    with zipfile.ZipFile(file_path, 'r') as zip_ref:
                        zip_ref.extractall(extracted_path)
                    print(f"Extracted: {filename}")
                except zipfile.BadZipFile:
                    print(f"Error: {filename} is not a valid zip file. Skipping.")
            else:
                print(f"Skipping {filename}, already extracted.")

# Step 5: Delete the downloaded folder containing .mglyph files
print("Deleting downloaded folder...")
shutil.rmtree(download_folder)
print("Downloaded folder deleted.")

print("Process complete. Extracted files are in:", extracted_folder)

# Experimental dataset with randomly colored stars creation

In this part we will be creating a folder containing the different pngs for the random colored stars that had been previously generated and we will write the json file that would contain the description of the dataset.

In [ ]:
import os
import json
import zipfile
import shutil
from datetime import datetime

If you suspect that the dataset has changed and more glypha were added please delete "extracted_files" and "Colored-Stars.zip" and rerun the previous notebook cells and the ones that would come after this markdown 

In [ ]:

# Define directories and output file
root_dir = 'extracted_files'
output_zip = 'data.zip'

# Check if the zip file already exists
if os.path.exists(output_zip):
    print(f"{output_zip} already exists. Skipping the process.")
    exit(0)

# Initialize dataset info
dataset_info = {
    "name": "Experimental dataset with randomly colored stars",
    "time-of-creation": datetime.now().strftime("%Y-%m-%d"),
    "samples": []
}

# Create ZIP file
with zipfile.ZipFile(output_zip, 'w') as zipf:
    for folder_name in os.listdir(root_dir):
        folder_path = os.path.join(root_dir, folder_name)
        metadata_path = os.path.join(folder_path, 'metadata.json')

        if os.path.isdir(folder_path) and os.path.exists(metadata_path):
            with open(metadata_path, 'r') as f:
                metadata = json.load(f)

            for old_file_name, value in metadata['images']:
                old_file_path = os.path.join(folder_path, old_file_name)
                new_file_name = f"{folder_name}-{old_file_name}"  # Rename files

                # Add file to dataset info
                dataset_info['samples'].append({"value": value, "file": new_file_name})

                # Write renamed file to ZIP
                zipf.write(old_file_path, new_file_name)

# Add dataset info JSON to ZIP
with zipfile.ZipFile(output_zip, 'a') as zipf:
    zipf.writestr('_dataset-info.json', json.dumps(dataset_info, indent=4))

print(f"Process completed. {output_zip} has been created.")